# Frontface-culling analysis on mirrorbench_v2 gt_geometry

For each image we compute:
- **(1)** non-black pixels inside the mirror in the original projection
- **(2)** non-black pixels inside the mirror in |original − frontface-culled|
- **(3)** ratio (2)/(1)

Then we plot histograms and browse samples sorted by (2) and (3) side by side.

In [1]:
import io, json, tempfile
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "..")

import yaml
from fill_my_mirror.storage import R2Client
from fill_my_mirror.blender import render_with_blender
from fill_my_mirror.projection.utils import load_rgb_image, load_binary_mask

with open("../configs/config.yaml") as f:
    config = yaml.safe_load(f)
BLENDER_PATH = Path("..") / config["blender_path"]

OUTPUT_DIR = Path("/tmp/fmc_stats")
OUTPUT_DIR.mkdir(exist_ok=True)
METRICS_PATH = OUTPUT_DIR / "metrics.json"

# A pixel is "non-black" if its brightest channel exceeds this value (0-255).
THRESHOLD = 10

r2 = R2Client()

def r2_bytes(key: str) -> bytes:
    return r2._s3.get_object(Bucket=r2._bucket, Key=key)["Body"].read()

def count_nonblack(img_rgb: np.ndarray, mask: np.ndarray) -> int:
    inside = img_rgb[mask]          # shape (N, 3)
    return int((inside.max(axis=1) > THRESHOLD).sum())

print("Setup complete.")

/home/ofek_basson/miniconda3/envs/fill-my-mirror/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


## Compute metrics (resumes from disk if partially done)

In [2]:
# Load previously saved metrics if available
if METRICS_PATH.exists():
    with open(METRICS_PATH) as f:
        results = json.load(f)
    print(f"Loaded {len(results)} cached results.")
else:
    results = {}

all_keys = sorted(
    k for k in r2.list_keys("mirrorbench_v2/gt_geometry/")
    if k.endswith("/projected_image.png")
)
print(f"Total images in mirrorbench_v2/gt_geometry/: {len(all_keys)}")
todo = [k for k in all_keys if k.split("/")[2] not in results]
print(f"Still to process: {len(todo)}")

Total images in mirrorbench_v2/gt_geometry/: 2991
Still to process: 2991


In [ ]:
for i, key in enumerate(todo):
    idx    = key.split("/")[2]
    prefix = f"mirrorbench_v2/gt_geometry/{idx}"
    print(f"[{i+1}/{len(todo)}] idx={idx}", end=" ", flush=True)

    orig_save = OUTPUT_DIR / f"{idx}_orig.png"
    fc_save   = OUTPUT_DIR / f"{idx}_fc.png"
    mask_save = OUTPUT_DIR / f"{idx}_mask.npy"

    glb_key  = f"{prefix}/reflected_scene.glb"
    npz_key  = f"{prefix}/blender_render_inputs.npz"
    img_key  = f"{prefix}/original_image.png"
    mask_key = f"{prefix}/generative_refinement_mask.png"

    missing = [k for k in (glb_key, npz_key, img_key, mask_key) if not r2.key_exists(k)]
    if missing:
        print(f"SKIP (missing: {Path(missing[0]).name})")
        continue

    # Download original projected_image once
    if not orig_save.exists():
        orig_save.write_bytes(r2_bytes(key))

    # Re-render with frontface culling (skip if already done)
    if not fc_save.exists():
        with tempfile.TemporaryDirectory() as tmp:
            tmp = Path(tmp)
            glb_path  = tmp / "reflected_scene.glb"
            npz_path  = tmp / "blender_render_inputs.npz"
            img_path  = tmp / "original_image.png"
            mask_path = tmp / "generative_refinement_mask.png"

            glb_path.write_bytes(r2_bytes(glb_key))
            npz_path.write_bytes(r2_bytes(npz_key))
            img_path.write_bytes(r2_bytes(img_key))
            mask_path.write_bytes(r2_bytes(mask_key))

            data = np.load(npz_path)
            intrinsics  = data["intrinsics"]
            image_shape = tuple(int(x) for x in data["image_shape"])

            raw_path = tmp / "raw_fc.png"
            bw_path  = tmp / "bw_fc.png"

            render_with_blender(
                blender_path=BLENDER_PATH,
                glb_path=glb_path,
                intrinsics=intrinsics,
                image_shape=image_shape,
                output_path=raw_path,
                bw_output_path=bw_path,
                tmp_dir=tmp,
                frontface_culling=True,
            )

            render_rgba = np.array(Image.open(raw_path).convert("RGBA"))
            render_rgb  = render_rgba[:, :, :3]
            alpha       = render_rgba[:, :, 3]

            original_scene = load_rgb_image(img_path)
            mirror         = load_binary_mask(mask_path)

            visible    = mirror & (alpha > 128)
            fc_img     = original_scene.copy()
            fc_img[visible] = render_rgb[visible]

            Image.fromarray(fc_img).save(fc_save)
            np.save(mask_save, mirror)

    # Compute metrics from saved files
    mirror   = np.load(mask_save)
    orig_arr = np.array(Image.open(orig_save).convert("RGB"))
    fc_arr   = np.array(Image.open(fc_save).convert("RGB"))
    diff_arr = np.abs(orig_arr.astype(int) - fc_arr.astype(int)).clip(0, 255).astype(np.uint8)

    m1 = count_nonblack(orig_arr, mirror)
    m2 = count_nonblack(diff_arr, mirror)
    m3 = m2 / m1 if m1 > 0 else 0.0

    results[idx] = {"m1": m1, "m2": m2, "m3": round(m3, 6)}
    print(f"m1={m1:6d}  m2={m2:6d}  m3={m3:.4f}")

    with open(METRICS_PATH, "w") as f:
        json.dump(results, f)

print(f"\nDone. {len(results)} images processed.")

[1/2991] idx=0 12:02:48 | INFO: Data are loaded, start creating Blender stuff
12:02:48 | INFO: Blender create Mesh node geometry_0
12:02:48 | INFO: glTF import finished in 0.15s
Blender 4.4.3 (hash 802179c51ccc built 2025-04-29 15:12:13)
Fra:1 Mem:91.28M (Peak 115.60M) | Time:00:01.15 | Rendering 1 / 64 samples
Fra:1 Mem:91.28M (Peak 115.60M) | Time:00:01.26 | Rendering 25 / 64 samples
Fra:1 Mem:91.28M (Peak 115.60M) | Time:00:01.34 | Rendering 50 / 64 samples
Fra:1 Mem:91.28M (Peak 115.60M) | Time:00:01.39 | Rendering 64 / 64 samples
Saved: '/tmp/tmpkppr9wua/raw_fc.png'
Time: 00:01.53 (Saving: 00:00.11)

Fra:1 Mem:90.20M (Peak 107.62M) | Time:00:00.05 | Rendering 1 / 64 samples
Fra:1 Mem:90.20M (Peak 107.62M) | Time:00:00.13 | Rendering 25 / 64 samples
Fra:1 Mem:90.20M (Peak 107.62M) | Time:00:00.20 | Rendering 50 / 64 samples
Fra:1 Mem:90.20M (Peak 107.62M) | Time:00:00.24 | Rendering 64 / 64 samples
Saved: '/tmp/tmpkppr9wua/bw_fc.png'
Time: 00:00.31 (Saving: 00:00.04)


Blender quit

## Histograms of (2) and (3)

In [ ]:
with open(METRICS_PATH) as f:
    results = json.load(f)

m1_vals = [v["m1"] for v in results.values()]
m2_vals = [v["m2"] for v in results.values()]
m3_vals = [v["m3"] for v in results.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(m2_vals, bins=40, color="steelblue", edgecolor="white")
axes[0].set_xlabel("(2) non-black pixels in |orig − fc| inside mirror")
axes[0].set_ylabel("Number of images")
axes[0].set_title("Distribution of metric (2)")

axes[1].hist(m3_vals, bins=40, color="coral", edgecolor="white")
axes[1].set_xlabel("(3) ratio (2) / (1)")
axes[1].set_ylabel("Number of images")
axes[1].set_title("Distribution of metric (3)")

plt.tight_layout()
plt.show()

print(f"(2): min={min(m2_vals)}  median={int(np.median(m2_vals))}  max={max(m2_vals)}")
print(f"(3): min={min(m3_vals):.4f}  median={np.median(m3_vals):.4f}  max={max(m3_vals):.4f}")

## Samples sorted by (2) vs sorted by (3)

Each row shows the same rank position from both sort orders:
left half = sorted by **(2)**, right half = sorted by **(3)**.

In [ ]:
N_SAMPLES = 10  # number of uniformly-spaced ranks to display

def uniform_sample(items, n):
    """Pick n uniformly-spaced entries from a sorted list."""
    if len(items) <= n:
        return items
    idx = np.linspace(0, len(items) - 1, n, dtype=int)
    return [items[i] for i in idx]

by_m2 = uniform_sample(sorted(results.items(), key=lambda x: x[1]["m2"]), N_SAMPLES)
by_m3 = uniform_sample(sorted(results.items(), key=lambda x: x[1]["m3"]), N_SAMPLES)

def load_triplet(idx):
    orig = np.array(Image.open(OUTPUT_DIR / f"{idx}_orig.png").convert("RGB"))
    fc   = np.array(Image.open(OUTPUT_DIR / f"{idx}_fc.png").convert("RGB"))
    diff = np.abs(orig.astype(int) - fc.astype(int)).clip(0, 255).astype(np.uint8)
    return orig, fc, diff

def col_label(idx, metrics):
    return f"idx={idx}\nm2={metrics['m2']}  m3={metrics['m3']:.3f}"

# 6 images per row: [orig|fc|diff] for m2-sorted  |  [orig|fc|diff] for m3-sorted
fig, axes = plt.subplots(N_SAMPLES, 6, figsize=(24, 4 * N_SAMPLES))

top_labels = ["Orig (↑m2)", "FC (↑m2)", "Diff (↑m2)",
              "Orig (↑m3)", "FC (↑m3)", "Diff (↑m3)"]
for col, lbl in enumerate(top_labels):
    axes[0, col].set_title(lbl, fontsize=11, fontweight="bold")

for row, ((idx2, met2), (idx3, met3)) in enumerate(zip(by_m2, by_m3)):
    triplet2 = load_triplet(idx2)
    triplet3 = load_triplet(idx3)

    for col, img in enumerate(list(triplet2) + list(triplet3)):
        axes[row, col].imshow(img)
        axes[row, col].axis("off")

    axes[row, 0].set_ylabel(col_label(idx2, met2), fontsize=8, rotation=0,
                            labelpad=60, va="center")
    axes[row, 3].set_ylabel(col_label(idx3, met3), fontsize=8, rotation=0,
                            labelpad=60, va="center")

    # Vertical divider between the two halves
    for ax in axes[row, :3]:
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[row, 2].spines["right"].set_visible(True)
    axes[row, 2].spines["right"].set_linewidth(2)
    axes[row, 2].spines["right"].set_color("gray")

plt.suptitle(
    f"Left: uniformly sampled from ranking by (2)   |   Right: uniformly sampled from ranking by (3)",
    fontsize=13, y=1.002
)
plt.tight_layout()
plt.show()